In [ ]:
!pip install torch torchvision scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from PIL import Image

In [ ]:
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'sqft': np.random.randint(500, 4000, n),
    'bedrooms': np.random.randint(1, 6, n),
    'bathrooms': np.random.randint(1, 4, n),
    'age': np.random.randint(1, 50, n),
    'price': np.random.randint(100000, 900000, n)
})

print(df.head())
print(df.shape)

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation')
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(df['price'], bins=30, kde=True, color='steelblue')
plt.title('Price Distribution')
plt.show()

In [ ]:
try:
    cnn_model = models.resnet18(weights=None)
except TypeError:
    cnn_model = models.resnet18(pretrained=False)

cnn_model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

feature_extractor = nn.Sequential(*list(cnn_model.children())[:-1])
feature_extractor.eval()

def extract_image_features(n_samples=200):
    features = []
    with torch.no_grad():
        for i in range(n_samples):
            dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
            img_tensor = transform(dummy_image).unsqueeze(0)
            feat = feature_extractor(img_tensor).squeeze().numpy()
            features.append(feat)
    return np.array(features)

print('Extracting CNN image features...')
image_features = extract_image_features(n)
print('Image features shape:', image_features.shape)

In [ ]:
tabular_features = df[['sqft', 'bedrooms', 'bathrooms', 'age']].values
scaler = StandardScaler()
tabular_scaled = scaler.fit_transform(tabular_features)

pca = PCA(n_components=10)
image_reduced = pca.fit_transform(image_features)

combined_features = np.hstack([tabular_scaled, image_reduced])
print('Combined features shape:', combined_features.shape)

In [ ]:
y = df['price'].values
X_train, X_test, y_train, y_test = train_test_split(combined_features, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('MAE:', round(mean_absolute_error(y_test, y_pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(y_test[:50], label='Actual', color='blue')
plt.plot(y_pred[:50], label='Predicted', color='orange')
plt.title('Actual vs Predicted House Prices (Multimodal)')
plt.xlabel('Samples')
plt.ylabel('Price')
plt.legend()
plt.tight_layout()
plt.show()